# Thuật toán AI: ID3 (Iterative Dichotomiser 3)

Notebook gồm 3 phần:
1. Sử dụng thư viện triển khai ID3.
2. Tự cài đặt thuật toán ID3.
3. Tạo dữ liệu giả lập và kiểm thử, so sánh hai cách triển khai.

**Lưu ý:** ID3 cơ bản làm việc với thuộc tính phân loại (categorical), chọn thuộc tính có Information Gain lớn nhất tại mỗi nút và không thực hiện pruning trong phiên bản nền tảng này.


## 1. Sử dụng thư viện

Sử dụng `chefboost` với cấu hình `ID3`. Thư viện này hỗ trợ nhiều thuật toán cây quyết định, trong đó có ID3.

Nếu môi trường chưa cài thư viện, chạy ô cài đặt bên dưới.


In [ ]:
%pip install -q chefboost pandas scikit-learn matplotlib

In [ ]:
import pandas as pd
import numpy as np
from chefboost import Chefboost as chef
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

RANDOM_STATE = 42


### 1.1. Hàm entropy và Information Gain để tham khảo

Entropy đo độ hỗn loạn của nhãn. Information Gain đo mức giảm entropy khi chia dữ liệu theo một thuộc tính.


In [ ]:
def entropy_from_series(y):
    probabilities = y.value_counts(normalize=True).to_numpy()
    return -np.sum(probabilities * np.log2(probabilities + 1e-12))

def information_gain_from_dataframe(df, feature, target):
    parent_entropy = entropy_from_series(df[target])
    weighted_child_entropy = 0.0

    for _, subset in df.groupby(feature):
        weighted_child_entropy += (
            len(subset) / len(df)
        ) * entropy_from_series(subset[target])

    return parent_entropy - weighted_child_entropy


### 1.2. Huấn luyện ID3 bằng Chefboost

Dữ liệu được tạo ở phần 3. Để notebook chạy tuần tự, phần huấn luyện bằng thư viện sẽ được thực hiện sau khi tạo dữ liệu và chia tập train/test.


In [ ]:
# Lệnh mẫu sẽ được thực hiện ở phần 3:
# config = {'algorithm': 'ID3'}
# model_lib = chef.fit(train_df.copy(), config=config, target_label='Play')
#
# Dự đoán một bản ghi:
# prediction = chef.predict(model_lib, test_df.iloc[0].drop('Play').tolist())


## 2. Tự cài đặt thuật toán ID3

Các bước chính:
1. Tính entropy của tập dữ liệu hiện tại.
2. Tính Information Gain của từng thuộc tính chưa sử dụng.
3. Chọn thuộc tính có Information Gain lớn nhất làm nút hiện tại.
4. Chia dữ liệu theo từng giá trị của thuộc tính đó và đệ quy.
5. Dừng khi nút thuần nhất, hết thuộc tính hoặc không còn dữ liệu để chia.

Khi gặp giá trị thuộc tính chưa xuất hiện trong quá trình huấn luyện, bộ phân loại trả về nhãn phổ biến nhất tại nút cha (default label).


In [ ]:
from collections import Counter
import math

def entropy(labels):
    counts = Counter(labels)
    total = len(labels)
    if total == 0:
        return 0.0
    return -sum((count / total) * math.log2(count / total)
                for count in counts.values())

def information_gain(data, feature, target):
    parent_entropy = entropy(data[target].tolist())
    total = len(data)
    weighted_entropy = 0.0

    for value, subset in data.groupby(feature, dropna=False):
        weighted_entropy += (len(subset) / total) * entropy(subset[target].tolist())

    return parent_entropy - weighted_entropy

def majority_class(labels):
    counts = Counter(labels)
    # Counter giữ thứ tự xuất hiện khi có trường hợp hòa.
    return counts.most_common(1)[0][0]

def build_id3_tree(data, features, target):
    labels = data[target].tolist()

    # Trường hợp 1: tất cả mẫu cùng một lớp.
    if len(set(labels)) == 1:
        return labels[0]

    # Trường hợp 2: không còn thuộc tính để chia.
    if not features:
        return majority_class(labels)

    gains = {feature: information_gain(data, feature, target)
             for feature in features}
    best_feature = max(gains, key=gains.get)

    # Không có lợi ích thông tin đáng kể: tạo lá theo lớp đa số.
    if gains[best_feature] <= 1e-12:
        return majority_class(labels)

    tree = {
        "feature": best_feature,
        "default": majority_class(labels),
        "branches": {}
    }

    remaining_features = [f for f in features if f != best_feature]

    for value, subset in data.groupby(best_feature, dropna=False):
        tree["branches"][value] = build_id3_tree(
            subset.drop(columns=[best_feature]),
            remaining_features,
            target
        )

    return tree

def predict_one(tree, sample):
    if not isinstance(tree, dict):
        return tree

    feature = tree["feature"]
    value = sample.get(feature)

    if value not in tree["branches"]:
        return tree["default"]

    return predict_one(tree["branches"][value], sample)

def predict_id3(tree, X):
    return [predict_one(tree, row) for row in X.to_dict(orient="records")]

def print_tree(tree, indent=""):
    if not isinstance(tree, dict):
        print(indent + "-> " + str(tree))
        return

    feature = tree["feature"]
    print(indent + f"[{feature}]")
    for value, child in tree["branches"].items():
        print(indent + f"  {feature} = {value}")
        print_tree(child, indent + "    ")


## 3. Data Mock và kiểm thử hai thuật toán

Tạo bộ dữ liệu mô phỏng quyết định có chơi thể thao ngoài trời hay không. Các thuộc tính đều là dữ liệu phân loại, phù hợp với ID3 cơ bản.

Sau đó chia dữ liệu thành tập huấn luyện và kiểm thử, huấn luyện cả hai mô hình và so sánh Accuracy.


In [ ]:
# Tạo dữ liệu mock có quy luật nhưng vẫn có một phần nhiễu.
rng = np.random.default_rng(RANDOM_STATE)
n_samples = 500

weather_values = ["Sunny", "Cloudy", "Rainy"]
temperature_values = ["Hot", "Mild", "Cool"]
humidity_values = ["High", "Normal"]
wind_values = ["Strong", "Weak"]

rows = []
for _ in range(n_samples):
    weather = rng.choice(weather_values, p=[0.38, 0.35, 0.27])
    temperature = rng.choice(temperature_values, p=[0.30, 0.45, 0.25])
    humidity = rng.choice(humidity_values, p=[0.48, 0.52])
    wind = rng.choice(wind_values, p=[0.35, 0.65])

    # Quy tắc giả lập để sinh nhãn.
    if weather == "Rainy":
        play = "No"
    elif weather == "Sunny" and humidity == "High":
        play = "No"
    elif weather == "Cloudy":
        play = "Yes"
    elif temperature == "Cool" and wind == "Weak":
        play = "Yes"
    else:
        play = "Yes" if rng.random() < 0.55 else "No"

    # Thêm nhiễu nhỏ để bài toán không hoàn toàn tách tuyến tính theo quy tắc.
    if rng.random() < 0.04:
        play = "No" if play == "Yes" else "Yes"

    rows.append({
        "Weather": weather,
        "Temperature": temperature,
        "Humidity": humidity,
        "Wind": wind,
        "Play": play
    })

df = pd.DataFrame(rows)
display(df.head(10))
print("Kích thước dữ liệu:", df.shape)
print("\nPhân bố nhãn:")
display(df["Play"].value_counts().to_frame("Count"))


In [ ]:
# Chia train/test có stratify để giữ tỷ lệ nhãn tương đối ổn định.
train_df, test_df = train_test_split(
    df,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=df["Play"]
)

X_train = train_df.drop(columns=["Play"])
y_train = train_df["Play"]
X_test = test_df.drop(columns=["Play"])
y_test = test_df["Play"]

print("Train:", train_df.shape, "| Test:", test_df.shape)


### 3.1. Kiểm thử thuật toán ID3 tự cài đặt

In [ ]:
features = X_train.columns.tolist()
tree_custom = build_id3_tree(train_df.copy(), features, target="Play")

print("Cây quyết định ID3 tự cài đặt:")
print_tree(tree_custom)

pred_custom = predict_id3(tree_custom, X_test)
acc_custom = accuracy_score(y_test, pred_custom)

print(f"\nAccuracy (ID3 tự viết): {acc_custom:.4f}")
print(classification_report(y_test, pred_custom, zero_division=0))


### 3.2. Kiểm thử ID3 bằng thư viện Chefboost

Chefboost nhận DataFrame có cả cột nhãn. Cấu hình `algorithm='ID3'` yêu cầu thư viện xây dựng cây theo ID3.


In [ ]:
config = {"algorithm": "ID3"}
model_lib = chef.fit(
    train_df.copy(),
    config=config,
    target_label="Play"
)

# Chefboost dự đoán theo một danh sách giá trị đầu vào theo thứ tự feature.
pred_lib = []
for _, row in X_test.iterrows():
    prediction = chef.predict(model_lib, row.tolist())
    # Chuẩn hóa kết quả dự đoán về nhãn đơn.
    if isinstance(prediction, (list, tuple, np.ndarray)):
        prediction = prediction[0]
    pred_lib.append(prediction)

acc_lib = accuracy_score(y_test, pred_lib)
print(f"Accuracy (Chefboost ID3): {acc_lib:.4f}")
print(classification_report(y_test, pred_lib, zero_division=0))


### 3.3. So sánh kết quả

In [ ]:
comparison = pd.DataFrame({
    "Algorithm": ["ID3 tự cài đặt", "Chefboost ID3"],
    "Accuracy": [acc_custom, acc_lib]
})
display(comparison)

ax = comparison.plot(
    x="Algorithm", y="Accuracy", kind="bar",
    ylim=(0, 1), legend=False, rot=0,
    title="So sánh Accuracy của hai cách triển khai"
)
ax.set_ylabel("Accuracy")
ax.grid(axis="y", alpha=0.3)
plt.show()


In [ ]:
# So sánh dự đoán trên một số mẫu đầu tiên.
sample_comparison = X_test.head(12).copy()
sample_comparison["Actual"] = y_test.head(12).values
sample_comparison["Custom_ID3"] = pred_custom[:12]
sample_comparison["Chefboost_ID3"] = pred_lib[:12]
display(sample_comparison)


### 3.4. Thử dự đoán một mẫu mới

Thay đổi các giá trị thuộc tính để quan sát kết quả dự đoán từ cây tự cài đặt.


In [ ]:
new_sample = {
    "Weather": "Cloudy",
    "Temperature": "Mild",
    "Humidity": "Normal",
    "Wind": "Weak"
}

custom_result = predict_one(tree_custom, new_sample)
library_result = chef.predict(
    model_lib,
    [new_sample[col] for col in X_train.columns]
)

print("Mẫu mới:", new_sample)
print("Dự đoán ID3 tự viết:", custom_result)
print("Dự đoán Chefboost ID3:", library_result)


## Kết luận và lưu ý

- Hai cách triển khai đều xây dựng cây quyết định dựa trên Information Gain.
- Kết quả có thể khác nhau do cách xử lý trường hợp hòa, giá trị thiếu, thứ tự thuộc tính, điều kiện dừng và chi tiết triển khai của thư viện.
- Accuracy chỉ phản ánh hiệu quả trên tập test được tạo trong lần chạy này; nên thay đổi `RANDOM_STATE` hoặc dùng cross-validation khi đánh giá nghiêm túc.
- ID3 cơ bản ưu tiên thuộc tính có nhiều giá trị nếu chúng tạo ra Information Gain lớn; thuật toán gốc không có pruning và không xử lý trực tiếp thuộc tính số liên tục nếu chưa rời rạc hóa.
